In [1]:
import json

In [17]:
def extract_subgraph(input_file, output_file, top_n=5):
    """
    Extracts a subgraph that:
    1. Preserves ALL nodes in the embedding layer (Layer 'E').
    2. Selects only the top N nodes for every other layer based on importance.
    3. Keeps edges only if they connect two selected nodes.
    
    Args:
        input_file (str): Path to the original JSON file.
        output_file (str): Path to save the filtered JSON file.
        top_n (int): Number of top nodes to keep per hidden layer.
    """
    print(f"Reading from {input_file}...")
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: File {input_file} not found.")
        return

    nodes = data.get('nodes', [])
    links = data.get('links', [])
    
    print(f"Original graph: {len(nodes)} nodes, {len(links)} links.")

    # 1. Group nodes by layer
    nodes_by_layer = {}
    for node in nodes:
        layer = str(node.get('layer'))
        if layer not in nodes_by_layer:
            nodes_by_layer[layer] = []
        nodes_by_layer[layer].append(node)

    # 2. Select nodes to keep
    selected_node_ids = set()
    
    for layer, layer_nodes in nodes_by_layer.items():
        # RULE 1: If layer is 'E' (Embedding), keep ALL nodes (Input Tokens)
        if layer == 'E':
            for node in layer_nodes:
                selected_node_ids.add(node['node_id'])
            continue

        # RULE 2: For all other layers, keep top N based on importance
        def get_importance(n):
            # Use absolute influence if available, else token_prob (for logits), else 0
            if n.get('influence') is not None:
                return abs(n['influence'])
            if n.get('token_prob') is not None:
                return n['token_prob']
            return 0.0

        # Sort descending by importance
        sorted_nodes = sorted(layer_nodes, key=get_importance, reverse=True)
        
        # Slice the top N
        top_k_nodes = sorted_nodes[:top_n]
        
        for node in top_k_nodes:
            selected_node_ids.add(node['node_id'])

    # 3. Filter the nodes list
    filtered_nodes = [n for n in nodes if n['node_id'] in selected_node_ids]

    # 4. Filter the links list
    # Only keep a link if BOTH source and target are in our selected set
    filtered_links = [
        l for l in links 
        if l['source'] in selected_node_ids and l['target'] in selected_node_ids
    ]

    # 5. Construct new data object
    new_data = data.copy()
    new_data['nodes'] = filtered_nodes
    new_data['links'] = filtered_links

    # 6. Save to output file
    print(f"Filtered graph: {len(filtered_nodes)} nodes, {len(filtered_links)} links.")
    print(f"Writing to {output_file}...")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(new_data, f, indent=2)

    print("Done.")

def update_metadata_file(metadata_path, new_slug, original_slug):
    """
    Duplicates an existing graph entry in the metadata file and saves it with a new slug.
    """
    try:
        with open(metadata_path, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
    except FileNotFoundError:
        print(f"Error: Metadata file {metadata_path} not found.")
        return

    graphs = metadata.get('graphs', [])
    
    # Find the original entry to copy from
    original_entry = next((g for g in graphs if g['slug'] == original_slug), None)
    
    if not original_entry:
        print(f"Error: Original slug '{original_slug}' not found in metadata.")
        return

    # Check if new slug already exists to avoid duplicates
    if any(g['slug'] == new_slug for g in graphs):
        print(f"Entry for '{new_slug}' already exists. Skipping update.")
        return

    # Create new entry
    new_entry = original_entry.copy()
    new_entry['slug'] = new_slug
    
    # Append and Save
    graphs.append(new_entry)
    metadata['graphs'] = graphs

    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Successfully added '{new_slug}' to {metadata_path}.")

def explain_graph(input_file, top_n=5):
    """
    Prints the most influential features for each layer.
    Note: The 'Text Interpretation' is not in the JSON, so this prints
    the Feature ID, which is the unique identifier for that concept.
    """
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: File {input_file} not found.")
        return

    nodes = data.get('nodes', [])
    
    # Group by layer
    nodes_by_layer = {}
    for node in nodes:
        layer = str(node.get('layer'))
        if layer not in nodes_by_layer:
            nodes_by_layer[layer] = []
        nodes_by_layer[layer].append(node)

    # Sort layers numerically for cleaner output
    # Handling 'E' (Embedding) by giving it index -1
    def layer_sort_key(k):
        if k == 'E': return -1
        if k == 'embedding': return -1
        try: return int(k)
        except: return 999

    sorted_layers = sorted(nodes_by_layer.keys(), key=layer_sort_key)

    print(f"{'LAYER':<10} | {'NODE/FEATURE ID':<20} | {'INFLUENCE':<15} | {'NOTES'}")
    print("-" * 70)

    for layer in sorted_layers:
        layer_nodes = nodes_by_layer[layer]
        
        # Sort by influence (absolute value)
        # Logits might use 'token_prob' instead
        def get_importance(n):
            if n.get('influence') is not None:
                return abs(n['influence'])
            if n.get('token_prob') is not None:
                return n['token_prob']
            return 0.0

        top_nodes = sorted(layer_nodes, key=get_importance, reverse=True)[:top_n]

        for node in top_nodes:
            # Identifier
            feat_id = node.get('feature', 'N/A')
            node_id = node.get('node_id', 'N/A')
            
            # Score
            inf = node.get('influence')
            if inf is None: inf = node.get('token_prob', 0.0)
            
            # "Clerp" holds text for the final output layer usually
            extra_info = node.get('clerp', '')
            
            # Special label for Embedding layer
            if layer == 'E':
                extra_info = f"Input Token (Ctx Idx {node.get('ctx_idx')})"

            print(f"{layer:<10} | {str(feat_id):<20} | {inf:<15.4f} | {extra_info}")
        
        print("-" * 70)


In [15]:
# --- Usage ---
if __name__ == "__main__":
    graph_dir = './graphs/'
    extract_subgraph(
        input_file=f'{graph_dir}/dallas-austin.json', 
        output_file=f'{graph_dir}/dallas-austin-subgraph-top3-E.json', 
        top_n=3
    )

Reading from ./graphs//dallas-austin.json...
Original graph: 1503 nodes, 350542 links.
Filtered graph: 89 nodes, 1078 links.
Writing to ./graphs//dallas-austin-subgraph-top3-E.json...
Done.


In [16]:
# --- Usage ---
graph_dir = './graphs/'
update_metadata_file(f'{graph_dir}/graph-metadata.json', 'dallas-austin-subgraph-top3-E', 'dallas-austin')

Successfully added 'dallas-austin-subgraph-top3-E' to ./graphs//graph-metadata.json.


In [20]:
explain_graph(f'{graph_dir}/dallas-austin-subgraph-top3-E.json', top_n=3)

LAYER      | NODE/FEATURE ID      | INFLUENCE       | NOTES
----------------------------------------------------------------------
E          | 3                    | 0.2600          | Input Token (Ctx Idx 3)
E          | 7                    | 0.2432          | Input Token (Ctx Idx 7)
E          | 4                    | 0.2215          | Input Token (Ctx Idx 4)
----------------------------------------------------------------------
0          | 108434900            | 0.7995          | 
0          | 46075199             | 0.7990          | 
0          | 23691285             | 0.7982          | 
----------------------------------------------------------------------
1          | 11488819             | 0.7999          | 
1          | 18207593             | 0.7998          | 
1          | 94277044             | 0.7977          | 
----------------------------------------------------------------------
2          | 97017                | 0.7993          | 
2          | 32332858             | 0